# Fine-tune FunctionGemma for banking tool routing

[Open in Colab](https://colab.research.google.com/github/LxYuan0420/nlp/blob/main/notebooks/Finetune_FunctionGemma_Banking77_Tool_Router_with_TRL_Colab.ipynb)

This notebook reproduces the remote-training idea from the Hugging Face article, but changes both the model and the task:

- **Model:** `google/functiongemma-270m-it`, a current 270M model designed to be specialized for function calling.
- **Data:** a balanced ten-intent slice of the maintained `mteb/banking77` Parquet dataset.
- **Task:** turn a customer message into exactly one structured banking support function call.
- **Measurement:** compare held-out tool-selection accuracy before and after four training epochs.
- **Logs:** TensorBoard inside Colab plus a persistent Trackio dashboard on Hugging Face.
- **Artifact:** publish a standalone full model, tokenizer, model card, metrics, predictions, and training history to the Hub.

The notebook is intentionally a walkthrough. One `uv` script owns data preparation, training, evaluation, tracking, model-card generation from recorded metrics, and Hub publication, so the teaching and one-command paths cannot drift apart.

### Verified reference run

On a free T4, the full 400-step run trained in 18 minutes 22 seconds and improved exact generated tool selection from **51% to 97%** on 100 deterministic held-out requests. Validation loss was lowest after epoch 2 (0.0428) and rose slightly by epoch 4 (0.0492), making the longer run useful for seeing where overfitting begins. The published FP32 checkpoint passed an end-to-end Hub reload test; forcing the weights to pure FP16 did not, so use the documented FP32 load path.

## 1. Before you start

Use **Runtime → Change runtime type → T4 GPU**. Free Colab GPU allocation is best effort, so a T4 may not always be available.

You also need to:

1. Accept the [FunctionGemma license](https://huggingface.co/google/functiongemma-270m-it).
2. Create a [Hugging Face token](https://huggingface.co/settings/tokens) with write access.
3. In Colab, open the key icon in the left sidebar and add a secret named `HF_TOKEN`. Enable notebook access for that secret.

The token is read into the process but never printed or stored in the notebook. The destination is determined by the token's Hugging Face username (`lxyuan` in the reference run), not by the Google account email used to open Colab.

In [ ]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN in Colab Secrets.")
os.environ["HF_TOKEN"] = hf_token
print("Hugging Face token loaded from Colab Secrets (value hidden).")

## 2. Install the tested training stack

The versions match the script's PEP 723 dependency block. TensorBoard is Google's free local experiment viewer; Trackio is Hugging Face's free local-first tracker and can persist the run in a Space.

In [ ]:
%pip install -q accelerate==1.14.0 datasets==5.0.1 'huggingface-hub>=1.4.0,<2' sentencepiece 'tensorboard==2.20.0' trackio==0.37.0 transformers==5.16.1 trl==1.12.0

## 3. Get the one-command experiment script

When this notebook is opened directly from GitHub, Colab receives only the notebook. This cell clones the repository so the exact same script powers both the notebook and command-line workflows.

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("/content/nlp")
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/LxYuan0420/nlp.git", str(repo_dir)],
        check=True,
    )

script_path = repo_dir / "scripts" / "finetune_functiongemma_banking77_colab.py"
if not script_path.is_file():
    raise FileNotFoundError(script_path)
print(f"Training script: {script_path}")

## 4. Understand the data structure

BANKING77 is a classification dataset. Its relevant source schema is:

```json
{
  "text": "string",
  "label": "integer class ID",
  "label_text": "human-readable class name"
}
```

For example: `text = "My card has not arrived"` and `label_text = "card_arrival"`. The script selects 80 training and 20 held-out examples for each of ten intents. It does not add a classification head. Instead, it turns the class label into the target assistant function call:

```json
{
  "messages": [
    {"role": "developer", "content": "You route customer requests by calling exactly one banking support tool."},
    {"role": "user", "content": "My card has not arrived"},
    {
      "role": "assistant",
      "tool_calls": [{
        "type": "function",
        "function": {
          "name": "handle_card_arrival",
          "arguments": {"customer_message": "My card has not arrived"}
        }
      }]
    }
  ],
  "tools": ["ten complete JSON function schemas"]
}
```

Each tool schema declares its function name, description, a required string argument named `customer_message`, and a string return type. FunctionGemma's chat template turns the structure into its native tool-declaration and function-call tokens. Run validation mode to print a real, fully formatted row.

In [ ]:
import subprocess

subprocess.run(["python", str(script_path), "--validate-only"], check=True)

## 5. Understand the loss and evaluation metric

This is supervised causal-language-model training. TRL 1.12 defaults to `chunked_nll`: the standard next-token cross-entropy (negative log-likelihood) computed in memory-saving chunks. At each position the contribution is `-log P(correct next token | previous tokens)`, and padding is ignored. See the [TRL 1.12 SFT documentation](https://huggingface.co/docs/trl/v1.12.0/en/sft_trainer).

For this verified conversational run, `assistant_only_loss=False`, so the loss covers every non-padding token in the rendered developer prompt, tool schemas, user message, and assistant function call. Exact tool-selection accuracy is different: it generates a response and checks whether the first complete function call names the expected handler. Accuracy is the application metric; cross-entropy is the differentiable training objective.

## 6. Start TensorBoard before training

Transformers writes loss, evaluation loss, learning rate, throughput, and epoch values to event files. Starting TensorBoard now lets its Scalars view update while the next cell trains. The same event files are uploaded with the final model.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/functiongemma-banking77-router/runs

## 7. Train, evaluate, and publish

The script first measures the untouched model, then fully fine-tunes all 270M parameters for four epochs and evaluates the final model on the same deterministic held-out sample. It saves weights, metrics, predictions, Trainer state, TensorBoard events, and local Trackio data. It then generates `README.md` directly from those observed metrics and publishes the complete output folder.

A full fine-tune is practical at this size on a T4 and leaves a standalone model. For larger models, switch this stage to LoRA or QLoRA.

In [ ]:
subprocess.run(["python", str(script_path)], check=True)

## 8. Inspect the result and varied tool calls

`training_metrics.json` contains the before/after predictions and exact configuration. `trainer_state.json` contains the step-by-step Trainer history. This cell prints several held-out inputs that selected different tools, followed by the two persistent Hugging Face links.

In [ ]:
import json
from IPython.display import Markdown, display

metrics_path = Path("/content/functiongemma-banking77-router/training_metrics.json")
metrics = json.loads(metrics_path.read_text())
print(f"Before: {metrics['baseline_tool_accuracy']:.2%}")
print(f"After:  {metrics['final_tool_accuracy']:.2%}")
print("\nDifferent inputs, different generated calls:")
seen_tools = set()
for row in metrics["final_predictions"]:
    if row["predicted_tool"] in seen_tools:
        continue
    seen_tools.add(row["predicted_tool"])
    print(f"\nInput:  {row['customer_message']}")
    print(f"Tool:   {row['predicted_tool']}")
    print(f"Output: {row.get('first_function_call', row.get('raw_generation'))}")
    if len(seen_tools) == 5:
        break
display(Markdown(
    f"- [Published model](https://huggingface.co/{metrics['model_repo_id']})\n"
    f"- [Trackio training dashboard](https://huggingface.co/spaces/{metrics['trackio_space_id']})"
))

## 9. Why did validation loss rise at the end?

Training loss fell from 0.0524 at epoch 1 to 0.0186 at epoch 4 because the model became increasingly confident on the 800 examples it repeatedly saw. Validation loss was best at epoch 2 (0.0428), then rose to 0.0492 while validation token accuracy still moved slightly upward. This is mild overfitting: a small number of confidently wrong unseen tokens can increase cross-entropy even when the count of correct tokens does not fall.

The final checkpoint was kept because it achieved 97/100 exact generated tool selections, the metric that matches the routing goal. We did not compute that generated metric at every epoch, so we cannot claim the epoch-2 checkpoint routes better. The next experiment should evaluate exact tool routing after every epoch and select by that metric.

The reference run also accidentally configured `warmup_steps=0.1`, effectively providing no meaningful warmup. Transformers 5.16 expects an integer step count, so the repository script now calculates 10% of optimizer steps (40 for the default run); this correction has not been presented as if it produced the already published numbers.

## What to try next

- Expand from ten to all 77 BANKING77 intents and measure where similar intents become confused.
- Compare full fine-tuning with LoRA using the same split, seed, and evaluation code.
- Add an out-of-scope tool or refusal path so unrelated requests are not forced into a banking handler.
- Replace public data with a reviewed domain dataset while preserving privacy and a real held-out test set.
- Measure not only tool selection but also whether arguments are copied and validated correctly.